In [2]:

import os
import glob

print("=== PAKDD-2027 CONTENTS ===")

base = "/kaggle/input"

for root, dirs, files in os.walk(base):
    if "pakdd" in root.lower():
        keras_files = [
            f for f in files if f.endswith(".keras")
        ]
        zip_files = [
            f for f in files if f.endswith(".zip")
        ]

        if keras_files or zip_files:
            print("\nFolder:", root)
            print("Keras models:", len(keras_files))
            print("ZIP files:", zip_files)
            print("Examples:", keras_files[:3])

print("\nTotal Keras checkpoints:",
      len(glob.glob(
          "/kaggle/input/**/*.keras",
          recursive=True
      )))


=== PAKDD-2027 CONTENTS ===

Folder: /kaggle/input/datasets/dmmehedihasanabid/pakdd-2027/models
Keras models: 30
ZIP files: []
Examples: ['EfficientNetB0_seed_123_source.keras', 'EfficientNetB0_seed_123_target_finetuned.keras', 'EfficientNetB0_seed_2024_source.keras']

Total Keras checkpoints: 30


In [3]:

import os
import glob
from collections import Counter

MODEL_DIR = "/kaggle/input/datasets/dmmehedihasanabid/pakdd-2027/models"

SOURCE_DIR = (
    "/kaggle/input/datasets/faysalmiah1721758/"
    "potato-dataset"
)

TARGET_DIR = (
    "/kaggle/input/datasets/shahadhossin567r7455/"
    "potato-leaf-disease-dataset/Potato Leaf DIsease"
)

SEEDS = [42, 123, 2024, 3407, 777]
MODELS = ["Lightweight_CNN", "MobileNetV2", "EfficientNetB0"]

# Verify all 30 required model files
missing = []

for model in MODELS:
    for seed in SEEDS:
        for stage in ["source", "target_finetuned"]:
            filename = f"{model}_seed_{seed}_{stage}.keras"
            path = os.path.join(MODEL_DIR, filename)

            if not os.path.isfile(path):
                missing.append(filename)

print("Missing checkpoints:", missing)
print("Checkpoint verification:",
      "PASS" if not missing else "FAIL")

# Verify original dataset directories
for name, directory in [
    ("SOURCE", SOURCE_DIR),
    ("TARGET", TARGET_DIR)
]:
    print(f"\n=== {name} DATASET ===")
    print("Directory exists:", os.path.isdir(directory))

    if os.path.isdir(directory):
        for folder in sorted(os.listdir(directory)):
            class_path = os.path.join(directory, folder)

            if os.path.isdir(class_path):
                count = len([
                    f for f in os.listdir(class_path)
                    if f.lower().endswith(
                        (".jpg", ".jpeg", ".png", ".bmp", ".webp")
                    )
                ])

                print(f"{folder}: {count} images")


Missing checkpoints: []
Checkpoint verification: PASS

=== SOURCE DATASET ===
Directory exists: True
Potato___Early_blight: 1000 images
Potato___Late_blight: 1000 images
Potato___healthy: 152 images

=== TARGET DATASET ===
Directory exists: True
Early Blight: 500 images
Fungal Diseases: 500 images
Healthy: 500 images
Late Blight: 500 images
Plant Pests: 500 images
Potato Cyst Nematode: 500 images
Potato Virus: 500 images


In [5]:

import os
import json
import zipfile
import tempfile
import shutil
import tensorflow as tf

MODEL_DIR = "/kaggle/input/datasets/dmmehedihasanabid/pakdd-2027/models"

def remove_null_quantization(obj):
    if isinstance(obj, dict):
        return {
            k: remove_null_quantization(v)
            for k, v in obj.items()
            if not (k == "quantization_config" and v is None)
        }
    if isinstance(obj, list):
        return [remove_null_quantization(v) for v in obj]
    return obj

def load_compatible_model(checkpoint):
    try:
        return tf.keras.models.load_model(
            checkpoint, compile=False
        )
    except (TypeError, ValueError) as error:
        if "quantization_config" not in str(error):
            raise

    temp_dir = tempfile.mkdtemp()
    repaired = os.path.join(temp_dir, "compatible.keras")

    try:
        with zipfile.ZipFile(checkpoint, "r") as source:
            with zipfile.ZipFile(repaired, "w") as destination:
                for item in source.infolist():
                    content = source.read(item.filename)

                    if item.filename == "config.json":
                        config = json.loads(content)
                        config = remove_null_quantization(config)
                        content = json.dumps(config).encode("utf-8")

                    destination.writestr(item, content)

        model = tf.keras.models.load_model(
            repaired, compile=False
        )
        return model

    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

# Test only one checkpoint
test_path = os.path.join(
    MODEL_DIR,
    "Lightweight_CNN_seed_42_source.keras"
)

model = load_compatible_model(test_path)

print("MODEL LOADED SUCCESSFULLY")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Parameters:", model.count_params())

del model
tf.keras.backend.clear_session()


MODEL LOADED SUCCESSFULLY
Input shape: (None, 256, 256, 3)
Output shape: (None, 3)
Parameters: 380739


In [9]:

import os, gc, json, glob
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.optimize import minimize_scalar
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobile_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficient_preprocess

MODEL_DIR = "/kaggle/input/datasets/dmmehedihasanabid/pakdd-2027/models"
SOURCE_DIR = "/kaggle/input/datasets/faysalmiah1721758/potato-dataset"
TARGET_DIR = "/kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease"

OUT = "/kaggle/working/CVPR2027_Temperature_Scaling"
os.makedirs(OUT, exist_ok=True)

SEEDS = [42, 123, 2024, 3407, 777]
MODELS = ["Lightweight_CNN", "MobileNetV2", "EfficientNetB0"]
CLASSES = ["Early Blight", "Late Blight", "Healthy"]

FOLDERS = {
    "source": [
        "Potato___Early_blight",
        "Potato___Late_blight",
        "Potato___healthy"
    ],
    "target": [
        "Early Blight",
        "Late Blight",
        "Healthy"
    ]
}

def collect_data(root, domain):
    paths, labels = [], []
    for label, folder in enumerate(FOLDERS[domain]):
        directory = os.path.join(root, folder)
        # Preserve original notebook's os.listdir ordering.
        for name in os.listdir(directory):
            if name.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp", ".webp")
            ):
                paths.append(os.path.join(directory, name))
                labels.append(label)
    return np.array(paths), np.array(labels)

source_x, source_y = collect_data(SOURCE_DIR, "source")
target_x, target_y = collect_data(TARGET_DIR, "target")

def split_data(paths, labels, seed):
    xtrain, xtemp, ytrain, ytemp = train_test_split(
        paths, labels, test_size=0.20,
        stratify=labels, random_state=seed
    )
    xval, xtest, yval, ytest = train_test_split(
        xtemp, ytemp, test_size=0.50,
        stratify=ytemp, random_state=seed
    )
    return xval, yval, xtest, ytest

def load_image(path, size, model_name):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(
        image, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (size, size))
    image = tf.cast(image, tf.float32)

    if model_name == "Lightweight_CNN":
        image = image / 255.0
    elif model_name == "MobileNetV2":
        image = mobile_preprocess(image)
    else:
        image = efficient_preprocess(image)

    return image

def predict_probs(model, paths, model_name):
    size = 256 if model_name == "Lightweight_CNN" else 224

    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(
        lambda p: load_image(p, size, model_name),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)

    return model.predict(ds, verbose=0)

def scale_probs(probs, temperature):
    probs = np.clip(
        np.asarray(probs, dtype=np.float64),
        1e-12, 1.0
    )
    logits = np.log(probs) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)

def nll(probs, labels):
    p = np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1)
    return float(-np.log(p).mean())

def fit_temperature(val_probs, val_labels):
    def objective(log_t):
        t = np.exp(log_t)
        return nll(scale_probs(val_probs, t), val_labels)

    result = minimize_scalar(
        objective,
        bounds=(np.log(0.05), np.log(20.0)),
        method="bounded",
        options={"xatol": 1e-5}
    )
    if not result.success:
        raise RuntimeError("Temperature optimization failed")
    return float(np.exp(result.x))

def ece(probs, labels, bins=15):
    confidence = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == labels)
    edges = np.linspace(0, 1, bins + 1)
    result = 0.0

    for i in range(bins):
        mask = (
            (confidence >= edges[i]) &
            ((confidence < edges[i+1]) if i < bins-1
             else (confidence <= edges[i+1]))
        )
        if mask.any():
            result += mask.mean() * abs(
                correct[mask].mean() - confidence[mask].mean()
            )
    return float(result)

def brier(probs, labels):
    onehot = np.eye(probs.shape[1])[labels]
    return float(np.mean(np.sum((probs-onehot)**2, axis=1)))

def metrics(probs, labels):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = pred == labels

    row = {
        "accuracy": float(correct.mean()),
        "nll": nll(probs, labels),
        "brier": brier(probs, labels),
        "ece_10": ece(probs, labels, 10),
        "ece_15": ece(probs, labels, 15),
        "ece_20": ece(probs, labels, 20)
    }

    # Match the original three-way threshold settings.
    for alpha, beta in [(0.95, 0.70),
                        (0.90, 0.60),
                        (0.85, 0.50)]:
        accept = conf >= alpha
        defer = (conf >= beta) & (conf < alpha)
        reject = conf < beta
        key = f"a{alpha}_b{beta}"

        row[f"{key}_accept_pct"] = float(accept.mean())
        row[f"{key}_defer_pct"] = float(defer.mean())
        row[f"{key}_reject_pct"] = float(reject.mean())
        row[f"{key}_accept_acc"] = (
            float(correct[accept].mean())
            if accept.any() else np.nan
        )
        row[f"{key}_wrong_accept_pct"] = float(
            (accept & ~correct).mean()
        )
    return row


rows = []

for model_name in MODELS:
    for seed in SEEDS:
        sxv, syv, sxt, syt = split_data(source_x, source_y, seed)
        txv, tyv, txt, tyt = split_data(target_x, target_y, seed)

        for stage in ["source", "target_finetuned"]:
            checkpoint = os.path.join(
                MODEL_DIR,
                f"{model_name}_seed_{seed}_{stage}.keras"
            )
            assert os.path.isfile(checkpoint), checkpoint

            if stage == "source":
                evaluations = [
                    ("source_3class", sxv, syv, sxt, syt),
                    ("zero_shot_target_3class", txv, tyv, txt, tyt)
                ]
            else:
                evaluations = [
                    ("finetuned_target_3class", txv, tyv, txt, tyt)
                ]

            # Skip loading the model if all evaluations are saved.
            pending = []
            for dataset_name, xv, yv, xt, yt in evaluations:
                prefix = f"{model_name}_seed_{seed}_{dataset_name}"
                result_file = os.path.join(OUT, prefix + ".json")

                if os.path.isfile(result_file):
                    with open(result_file) as f:
                        rows.extend(json.load(f))
                    print("SKIP:", prefix, flush=True)
                else:
                    pending.append(
                        (dataset_name, xv, yv, xt, yt, result_file)
                    )

            if not pending:
                continue

            tf.keras.backend.clear_session()
            model = load_compatible_model(checkpoint)

            for dataset_name, xv, yv, xt, yt, result_file in pending:
                prefix = f"{model_name}_seed_{seed}_{dataset_name}"
                print("RUN:", prefix, flush=True)

                val_probs = predict_probs(model, xv, model_name)
                test_probs = predict_probs(model, xt, model_name)

                T = fit_temperature(val_probs, yv)
                calibrated = scale_probs(test_probs, T)

                result_rows = []

                for condition, probs in [
                    ("before", test_probs),
                    ("after", calibrated)
                ]:
                    row = {
                        "model": model_name,
                        "seed": seed,
                        "dataset": dataset_name,
                        "condition": condition,
                        "temperature": T,
                        "n_val": len(yv),
                        "n_test": len(yt)
                    }

                    row.update(metrics(probs, yt))
                    result_rows.append(row)

                with open(result_file, "w") as f:
                    json.dump(result_rows, f, indent=2)

                rows.extend(result_rows)

                pd.DataFrame(rows).to_csv(
                    os.path.join(OUT, "calibration_per_seed.csv"),
                    index=False
                )

                print(
                    f"T={T:.4f} | "
                    f"ECE before={result_rows[0]['ece_15']:.4f} | "
                    f"after={result_rows[1]['ece_15']:.4f}",
                    flush=True
                )

            del model
            gc.collect()
            tf.keras.backend.clear_session()

results = pd.DataFrame(rows)

results.to_csv(
    os.path.join(OUT, "calibration_per_seed.csv"),
    index=False
)

summary = results.groupby(
    ["model", "dataset", "condition"],
    as_index=False
).agg(
    accuracy_mean=("accuracy", "mean"),
    ece15_mean=("ece_15", "mean"),
    ece15_std=("ece_15", "std"),
    nll_mean=("nll", "mean"),
    brier_mean=("brier", "mean"),
    temperature_mean=("temperature", "mean")
)

summary.to_csv(
    os.path.join(OUT, "calibration_summary.csv"),
    index=False
)

print("\nCOMPLETED EVALUATIONS:", len(results) // 2)
print("Expected: 45")

display(summary)


RUN: Lightweight_CNN_seed_42_source_3class


I0000 00:00:1793754304.522903     136 service.cc:152] XLA service 0x7b938c106070 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793754304.522954     136 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1793754304.522962     136 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1793754304.842382     136 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1793754307.613046     136 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


T=0.6204 | ECE before=0.0851 | after=0.0317
RUN: Lightweight_CNN_seed_42_zero_shot_target_3class
T=1.8752 | ECE before=0.1639 | after=0.1004
RUN: Lightweight_CNN_seed_42_finetuned_target_3class
T=0.7713 | ECE before=0.0611 | after=0.0587
RUN: Lightweight_CNN_seed_123_source_3class
T=0.7443 | ECE before=0.0617 | after=0.0396
RUN: Lightweight_CNN_seed_123_zero_shot_target_3class
T=2.6963 | ECE before=0.2071 | after=0.1157
RUN: Lightweight_CNN_seed_123_finetuned_target_3class
T=0.9540 | ECE before=0.0597 | after=0.0631
RUN: Lightweight_CNN_seed_2024_source_3class
T=0.6565 | ECE before=0.0784 | after=0.0448
RUN: Lightweight_CNN_seed_2024_zero_shot_target_3class
T=1.9342 | ECE before=0.1304 | after=0.0644
RUN: Lightweight_CNN_seed_2024_finetuned_target_3class
T=0.6696 | ECE before=0.0642 | after=0.0588
RUN: Lightweight_CNN_seed_3407_source_3class
T=0.5085 | ECE before=0.0533 | after=0.0570
RUN: Lightweight_CNN_seed_3407_zero_shot_target_3class
T=2.5097 | ECE before=0.1570 | after=0.1019
RUN

2026-11-04 01:06:12.572721: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:06:12.710314: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:06:24.436439: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:06:24.574008: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:06:34.873642: E external/local_xla/xla/stream_

T=0.0693 | ECE before=0.0045 | after=0.0000
RUN: MobileNetV2_seed_42_zero_shot_target_3class


2026-11-04 01:06:45.175175: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:06:45.325539: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:06:45.462869: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


... (2 outputs left out)

T=0.8010 | ECE before=0.0138 | after=0.0078
RUN: EfficientNetB0_seed_42_zero_shot_target_3class


2026-11-04 01:09:31.732518: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:09:31.874064: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:09:32.202453: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:09:32.344672: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-11-04 01:09:32.485362: E external/local_xla/xla/stream_

T=0.8829 | ECE before=0.0552 | after=0.0465
RUN: EfficientNetB0_seed_42_finetuned_target_3class
T=0.2675 | ECE before=0.0174 | after=0.0113
RUN: EfficientNetB0_seed_123_source_3class
T=0.0500 | ECE before=0.0142 | after=0.0085
RUN: EfficientNetB0_seed_123_zero_shot_target_3class
T=0.6562 | ECE before=0.0581 | after=0.0370
RUN: EfficientNetB0_seed_123_finetuned_target_3class
T=0.0501 | ECE before=0.0136 | after=0.0133
RUN: EfficientNetB0_seed_2024_source_3class
T=0.0500 | ECE before=0.0101 | after=0.0093
RUN: EfficientNetB0_seed_2024_zero_shot_target_3class
T=0.8254 | ECE before=0.0619 | after=0.0684
RUN: EfficientNetB0_seed_2024_finetuned_target_3class
T=0.0500 | ECE before=0.0066 | after=0.0000
RUN: EfficientNetB0_seed_3407_source_3class
T=0.8759 | ECE before=0.0174 | after=0.0125
RUN: EfficientNetB0_seed_3407_zero_shot_target_3class
T=0.6072 | ECE before=0.0610 | after=0.0649
RUN: EfficientNetB0_seed_3407_finetuned_target_3class
T=0.5737 | ECE before=0.0135 | after=0.0057
RUN: Effici

,model,dataset,condition,accuracy_mean,ece15_mean,ece15_std,nll_mean,brier_mean,temperature_mean
0,EfficientNetB0,finetuned_target_3class,after,0.994667,0.006060,0.006198,0.056770,0.009631,0.198261
1,EfficientNetB0,finetuned_target_3class,before,0.994667,0.011378,0.004989,0.015074,0.006700,0.198261
2,EfficientNetB0,source_3class,after,0.993519,0.010083,0.002176,0.045511,0.009940,0.524271
3,EfficientNetB0,source_3class,before,0.993519,0.014521,0.002991,0.022688,0.008435,0.524271
4,EfficientNetB0,zero_shot_target_3class,after,0.910667,0.050058,0.015926,0.227942,0.129920,0.786925
5,EfficientNetB0,zero_shot_target_3class,before,0.910667,0.054136,0.011270,0.228145,0.128789,0.786925
6,Lightweight_CNN,finetuned_target_3class,after,0.853333,0.061527,0.010479,0.351527,0.203197,0.805170
7,Lightweight_CNN,finetuned_target_3class,before,0.853333,0.066333,0.012522,0.352397,0.201187,0.805170
8,Lightweight_CNN,source_3class,after,0.913889,0.039115,0.013041,0.281435,0.139073,0.653100
9,Lightweight_CNN,source_3class,before,0.913889,0.069502,0.012730,0.280968,0.146825,0.653100


In [10]:

import shutil

shutil.make_archive(
    "/kaggle/working/CVPR2027_Temperature_Scaling",
    "zip",
    "/kaggle/working/CVPR2027_Temperature_Scaling"
)

print("Calibration results saved.")


Calibration results saved.


In [11]:

import os
import glob

RESNET_DIR = "/kaggle/input/datasets/dmmehedihasanabid/pakdd-2027-2"

resnet_files = sorted(
    p for p in glob.glob(
        os.path.join(RESNET_DIR, "**", "*.keras"),
        recursive=True
    )
    if "resnet50" in os.path.basename(p).lower()
)

print("ResNet-50 checkpoints found:", len(resnet_files))

for path in resnet_files:
    print(os.path.relpath(path, RESNET_DIR))

assert len(resnet_files) == 10, (
    "Expected 10 ResNet-50 checkpoints."
)


ResNet-50 checkpoints found: 10
models/ResNet50_seed_123_source.keras
models/ResNet50_seed_123_target_finetuned.keras
models/ResNet50_seed_2024_source.keras
models/ResNet50_seed_2024_target_finetuned.keras
models/ResNet50_seed_3407_source.keras
models/ResNet50_seed_3407_target_finetuned.keras
models/ResNet50_seed_42_source.keras
models/ResNet50_seed_42_target_finetuned.keras
models/ResNet50_seed_777_source.keras
models/ResNet50_seed_777_target_finetuned.keras


In [13]:

from tensorflow.keras.applications.resnet50 import (
    preprocess_input as resnet_preprocess
)

import tensorflow as tf

RESNET_MODEL_DIR = (
    "/kaggle/input/datasets/"
    "dmmehedihasanabid/pakdd-2027-2/models"
)

checkpoint = (
    RESNET_MODEL_DIR +
    "/ResNet50_seed_42_source.keras"
)

model = load_compatible_model(checkpoint)

print("ResNet-50 loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Parameters:", model.count_params())

del model
tf.keras.backend.clear_session()


ResNet-50 loaded successfully!
Input shape: (None, 224, 224, 3)
Output shape: (None, 3)
Parameters: 23593859


In [14]:

import os
import gc
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications.resnet50 import (
    preprocess_input as resnet_preprocess
)

RESNET_MODEL_DIR = (
    "/kaggle/input/datasets/"
    "dmmehedihasanabid/pakdd-2027-2/models"
)

RESNET_OUT = "/kaggle/working/CVPR2027_ResNet50_Calibration"
os.makedirs(RESNET_OUT, exist_ok=True)

# ResNet-50 preprocessing
def predict_resnet(model, paths):

    def preprocess(path):
        image = tf.io.read_file(path)
        image = tf.image.decode_image(
            image,
            channels=3,
            expand_animations=False
        )
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, (224, 224))
        image = tf.cast(image, tf.float32)

        return resnet_preprocess(image)

    dataset = tf.data.Dataset.from_tensor_slices(paths)

    dataset = dataset.map(
        preprocess,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(32).prefetch(tf.data.AUTOTUNE)

    return model.predict(dataset, verbose=0)


# Resume existing ResNet results if available
rows = []

for seed in SEEDS:

    sxv, syv, sxt, syt = split_data(
        source_x, source_y, seed
    )

    txv, tyv, txt, tyt = split_data(
        target_x, target_y, seed
    )

    for stage in ["source", "target_finetuned"]:

        checkpoint = os.path.join(
            RESNET_MODEL_DIR,
            f"ResNet50_seed_{seed}_{stage}.keras"
        )

        if stage == "source":
            evaluations = [
                ("source_3class", sxv, syv, sxt, syt),
                ("zero_shot_target_3class", txv, tyv, txt, tyt)
            ]
        else:
            evaluations = [
                ("finetuned_target_3class", txv, tyv, txt, tyt)
            ]

        pending = []

        for dataset_name, xv, yv, xt, yt in evaluations:

            prefix = f"ResNet50_seed_{seed}_{dataset_name}"

            result_file = os.path.join(
                RESNET_OUT,
                prefix + ".json"
            )

            if os.path.exists(result_file):
                with open(result_file) as f:
                    rows.extend(json.load(f))

                print("SKIP:", prefix, flush=True)
            else:
                pending.append(
                    (dataset_name, xv, yv, xt, yt, result_file)
                )

        if not pending:
            continue

        tf.keras.backend.clear_session()

        model = load_compatible_model(checkpoint)

        for dataset_name, xv, yv, xt, yt, result_file in pending:

            prefix = f"ResNet50_seed_{seed}_{dataset_name}"

            print("RUN:", prefix, flush=True)

            val_probs = predict_resnet(model, xv)
            test_probs = predict_resnet(model, xt)

            T = fit_temperature(val_probs, yv)

            calibrated_probs = scale_probs(test_probs, T)

            result_rows = []

            for condition, probs in [
                ("before", test_probs),
                ("after", calibrated_probs)
            ]:

                row = {
                    "model": "ResNet50",
                    "seed": seed,
                    "dataset": dataset_name,
                    "condition": condition,
                    "temperature": T,
                    "n_val": len(yv),
                    "n_test": len(yt)
                }

                row.update(metrics(probs, yt))
                result_rows.append(row)

            # Save immediately after each evaluation
            with open(result_file, "w") as f:
                json.dump(result_rows, f, indent=2)

            rows.extend(result_rows)

            pd.DataFrame(rows).to_csv(
                os.path.join(
                    RESNET_OUT,
                    "resnet_calibration_per_seed.csv"
                ),
                index=False
            )

            print(
                f"Temperature: {T:.4f} | "
                f"ECE before: {result_rows[0]['ece_15']:.4f} | "
                f"ECE after: {result_rows[1]['ece_15']:.4f}",
                flush=True
            )

        del model
        gc.collect()
        tf.keras.backend.clear_session()


results = pd.DataFrame(rows)

summary = results.groupby(
    ["model", "dataset", "condition"],
    as_index=False
).agg(
    accuracy_mean=("accuracy", "mean"),
    ece15_mean=("ece_15", "mean"),
    ece15_std=("ece_15", "std"),
    nll_mean=("nll", "mean"),
    brier_mean=("brier", "mean"),
    temperature_mean=("temperature", "mean")
)

summary.to_csv(
    os.path.join(
        RESNET_OUT,
        "resnet_calibration_summary.csv"
    ),
    index=False
)

print("\nCompleted ResNet evaluations:", len(results) // 2)
print("Expected: 15")

display(summary)


RUN: ResNet50_seed_42_source_3class
Temperature: 1.2207 | ECE before: 0.0196 | ECE after: 0.0222
RUN: ResNet50_seed_42_zero_shot_target_3class
Temperature: 2.0924 | ECE before: 0.1135 | ECE after: 0.0606
RUN: ResNet50_seed_42_finetuned_target_3class
Temperature: 1.1623 | ECE before: 0.0259 | ECE after: 0.0221
RUN: ResNet50_seed_123_source_3class
Temperature: 1.1087 | ECE before: 0.0181 | ECE after: 0.0183
RUN: ResNet50_seed_123_zero_shot_target_3class
Temperature: 3.9856 | ECE before: 0.1064 | ECE after: 0.0729
RUN: ResNet50_seed_123_finetuned_target_3class
Temperature: 1.8041 | ECE before: 0.0186 | ECE after: 0.0223
RUN: ResNet50_seed_2024_source_3class
Temperature: 0.0500 | ECE before: 0.0078 | ECE after: 0.0077
RUN: ResNet50_seed_2024_zero_shot_target_3class
Temperature: 4.7719 | ECE before: 0.0904 | ECE after: 0.0518
RUN: ResNet50_seed_2024_finetuned_target_3class
Temperature: 0.7495 | ECE before: 0.0133 | ECE after: 0.0138
RUN: ResNet50_seed_3407_source_3class
Temperature: 1.5170 

,model,dataset,condition,accuracy_mean,ece15_mean,ece15_std,nll_mean,brier_mean,temperature_mean
0,ResNet50,finetuned_target_3class,after,0.977333,0.019545,0.009519,0.047683,0.028392,1.267202
1,ResNet50,finetuned_target_3class,before,0.977333,0.018905,0.008941,0.046179,0.029376,1.267202
2,ResNet50,source_3class,after,0.983333,0.015231,0.005863,0.055308,0.027107,1.076657
3,ResNet50,source_3class,before,0.983333,0.015125,0.004620,0.057024,0.026344,1.076657
4,ResNet50,zero_shot_target_3class,after,0.874667,0.062190,0.008411,0.385135,0.197237,3.998365
5,ResNet50,zero_shot_target_3class,before,0.874667,0.112071,0.020005,0.793708,0.228213,3.998365


In [15]:

import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/CVPR2027_ResNet50_Calibration",
    "zip",
    "/kaggle/working/CVPR2027_ResNet50_Calibration"
)

print("Saved:", zip_path)


Saved: /kaggle/working/CVPR2027_ResNet50_Calibration.zip
